# Corporate Credit Rating Classification

## A leakage-aware public-data benchmark

This concise report presents two tasks: investment-grade versus speculative-grade classification, followed by a seven-band rating extension. All transformation and modeling logic lives in the tested `credit_rating` package; this notebook is the presentation layer.

## Research questions

1. Can historical financial ratios distinguish investment-grade from speculative-grade observations?
2. How much performance remains when the target is expanded to seven ordered rating bands?
3. How different are chronological results from performance on issuers excluded from a training fold?

In [ ]:
import json
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
metrics_dir = PROJECT_ROOT / 'outputs' / 'metrics'

audit = json.loads((metrics_dir / 'data_audit.json').read_text())
binary = json.loads((metrics_dir / 'binary_logistic.json').read_text())
seven_band = json.loads((metrics_dir / 'seven_band_random_forest.json').read_text())

## Data audit

- **7,805** historical observations from **2010–2016**
- **686** unique CIK identifiers
- **23** original rating labels consolidated into **7** ordered bands
- Source `Binary Rating` matches the broad target and is therefore excluded from predictors
- **850** available predictor vectors map to more than one observed rating, demonstrating label ambiguity

In [ ]:
pd.Series({
    'Rows': audit['rows'],
    'Unique CIK': audit['issuers']['unique_cik'],
    'Original rating classes': audit['ratings']['original_classes'],
    'Ambiguous predictor vectors': audit['label_ambiguity']['predictor_vectors_with_multiple_ratings'],
}, name='Data audit')

## Target distribution

![Seven-band target distribution](../outputs/charts/seven_band_distribution.png)

## Validation design

- **Training:** 2010–2014
- **Model selection:** 2015 only
- **Final chronological holdout:** 2016
- **Robustness check:** five-fold stratified group validation by CIK

Corporation name, CIK, ticker, rating date, original rating, and source binary rating are excluded from the feature matrix. Imputation, one-hot encoding, and scaling are fitted inside each pipeline.

In [ ]:
pd.DataFrame([
    {
        'Task': 'Binary',
        'Model': binary['model'],
        'Balanced accuracy': binary['balanced_accuracy'],
        'Macro-F1': binary['macro_f1'],
        'Grouped CV balanced accuracy': binary['unseen_issuer_group_cv']['balanced_accuracy_mean'],
    },
    {
        'Task': 'Seven band',
        'Model': seven_band['model'],
        'Balanced accuracy': seven_band['balanced_accuracy'],
        'Macro-F1': seven_band['macro_f1'],
        'Grouped CV balanced accuracy': seven_band['unseen_issuer_group_cv']['balanced_accuracy_mean'],
    },
]).round(3)

## Verified results

| Task | Selected model | 2016 balanced accuracy | 2016 macro-F1 |
|---|---|---:|---:|
| Investment grade vs. speculative grade | Logistic regression | 0.788 | 0.777 |
| Seven ordered rating bands | Random forest | 0.451 | 0.405 |

The binary model identifies **87.7%** of speculative-grade observations in the 2016 holdout and reaches **0.881 ROC-AUC**. The grouped estimate is lower, which is the more cautious result for issuers unseen during training.

### Binary holdout

![Binary confusion matrix](../outputs/charts/binary_logistic_confusion_matrix.png)

### Seven-band holdout

![Seven-band confusion matrix](../outputs/charts/seven_band_random_forest_confusion_matrix.png)

## Interpretation and limitations

The binary task is the defensible headline result. The seven-band model is retained as an extension: 82.8% of its predictions are within one broad band, but issuer-grouped macro-F1 is only 0.353. This gap is important evidence that the public ratios do not reproduce a full agency rating process.

The data is historical, agency methodology is not fully documented, and qualitative or forward-looking credit information is absent. The benchmark is educational and must not be used for lending, investment, or current issuer-rating decisions.